In [1]:
!git clone https://github.com/masamune-prog/DeepSC.git

Cloning into 'DeepSC'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 122 (delta 57), reused 107 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 4.98 MiB | 19.12 MiB/s, done.
Resolving deltas: 100% (57/57), done.


In [2]:
import os

# Change directory to DeepSC for all subsequent commands in this cell
os.chdir('/kaggle/working/DeepSC')

In [3]:
import os, math, time, json, random
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
import json
from datasets import load_dataset

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

torch: 2.10.0+cu128
cuda available: True
GPU: Tesla T4
bf16 supported: True
cuda


In [4]:
!uv venv
!source .venv/bin/activate
!pip install -r requirements.txt

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.8/377.8 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 3.2 MB/s eta 0:00:00
  Created wheel for bert4keras: filename=bert4keras-0.11.5-py3-none-any.whl size=52116 sha256=c8fecca804638a07b08f4ab6d795cbd2caa0d393e98476608f0680d5e58c05e7
  Stored in directory: /root/.cache/pip/wheels/15/94/40/a98727691205878b00aaf427d5448a646bd8f9cb293755c60d
Successfully built bert4keras
  Attempting uninstall: keras
    Found existing installation: keras 3.13.2
    Uninstalling keras-3.13.2:
      Successfully uninstalled keras-3.13.2
ERROR: pip's dependency resolver does not currently take int

In [5]:
!uv pip install bert-score
!uv pip install tokenizers

Using Python 3.12.13 environment at: /usr
Resolved 69 packages in 525ms                                        
Prepared 1 package in 34ms                                               
Installed 1 package in 7ms                                  
 + bert-score==0.3.13
Using Python 3.12.13 environment at: /usr
Checked 1 package in 130ms


In [6]:
%%capture
# Now create data directory, download, extract, and preprocess within DeepSC
!mkdir -p data
!wget http://www.statmt.org/europarl/v7/europarl.tgz
!tar zxvf europarl.tgz
!python preprocess_text.py

In [24]:
import os
import re
import pickle
import unicodedata
from tqdm import tqdm
from w3lib.html import remove_tags
from dataclasses import dataclass
# Hugging Face Tokenizer imports
from tokenizers import Tokenizer, normalizers, pre_tokenizers, processors
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer

# --- CONFIGURATION PATHS ---
INPUT_DATA_DIR = 'txt/en'
OUTPUT_TRAIN_DIR = 'txt/train_data.pkl'
OUTPUT_TEST_DIR = 'txt/test_data.pkl'
OUTPUT_VOCAB_PATH = 'txt/vocab.json'

# Ensure the directories exist
os.makedirs(INPUT_DATA_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_TRAIN_DIR), exist_ok=True)

In [8]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.normalizers import NFKC
from tokenizers.processors import TemplateProcessing


In [9]:
def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

def normalize_string(s):
    # Normalize unicode characters
    s = unicode_to_ascii(s)
    # Remove XML/HTML tags
    s = remove_tags(s)
    # Add white space before !.?
    s = re.sub(r'([!.?])', r' \1', s)
    # Strip everything except letters and base punctuation
    s = re.sub(r'[^a-zA-Z.!?]+', r' ', s)
    s = re.sub(r'\s+', r' ', s)
    return s.strip().lower()

def cut_data(cleaned, min_length=4, max_length=30):
    cutted_lines = []
    for line in cleaned:
        length = len(line.split())
        if min_length < length < max_length:
            cutted_lines.append(line)
    return cutted_lines

In [10]:

# --- LOAD AND PREPROCESS ---
sentences = []
print('Processing raw files...')
for fn in tqdm(os.listdir(INPUT_DATA_DIR)):
    if not fn.endswith('.txt'): 
        continue
    with open(os.path.join(INPUT_DATA_DIR, fn), 'r', encoding='utf8') as f:
        raw_data = f.read()
    
    file_sentences = raw_data.strip().split('\n')
    cleaned_sentences = [normalize_string(s) for s in file_sentences]
    cleaned_sentences = cut_data(cleaned_sentences)
    sentences.extend(cleaned_sentences)

# Deduplicate
sentences = list(dict.fromkeys(sentences))
print(f"\nSuccessfully loaded {len(sentences)} unique sentences.")
print("Sample clean text:", sentences[:3])

Processing raw files...


100%|██████████| 9672/9672 [01:33<00:00, 103.57it/s]


Successfully loaded 73472 unique sentences.
Sample clean text: ['common organisation of agricultural markets and specific provisions for certain agricultural products as regards the national quotas for milk debate', 'the agreed compromise has two focal points . commissioner you have said as much . i have a different opinion on this .', 'we shall continue to discuss the choice of the best solution in the future .']


In [11]:
# 1. Initialize WordLevel Model
tokenizer = Tokenizer(WordLevel(unk_token="<UNK>"))

# 2. Add Normalization & Pre-tokenization Rules
tokenizer.normalizer = normalizers.Sequence([
    normalizers.NFKD(),
    normalizers.Lowercase()
])

tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.WhitespaceSplit(),
    pre_tokenizers.Punctuation()
])

# 3. Train the Tokenizer
# The special tokens array maps them precisely to indices 0, 1, 2, 3 as in your original SPECIAL_TOKENS dict
trainer = WordLevelTrainer(
    min_frequency=1,
    special_tokens=["<PAD>", "<START>", "<END>", "<UNK>"]
)

print("Training Tokenizer...")
tokenizer.train_from_iterator(sentences, trainer=trainer)

# 4. Attach the Template Post-Processor (Inserts <START> and <END> dynamically)
start_id = tokenizer.token_to_id("<START>")
end_id = tokenizer.token_to_id("<END>")

tokenizer.post_processor = processors.TemplateProcessing(
    single="<START> $A <END>",
    special_tokens=[
        ("<START>", start_id),
        ("<END>", end_id),
    ],
)

# 5. Save the trained model config
tokenizer.save(OUTPUT_VOCAB_PATH)
print(f"Tokenizer saved to: {OUTPUT_VOCAB_PATH}")

Training Tokenizer...
Tokenizer saved to: txt/vocab.json


In [12]:
from transformers import PreTrainedTokenizerFast
hf_tokenizer = PreTrainedTokenizerFast(tokenizer_file="txt/vocab.json")

In [13]:
print("Example encoding:", hf_tokenizer.encode("Hello world!")[:20])

Example encoding: [1, 17692, 241, 174, 2]


In [14]:
# =======================
# 0) Choose a run profile
# =======================

RUN_MODE = "budget_100"   # "quick" or "budget_100"

# You can always override individual values later.

if RUN_MODE == "quick":
    # Small + fast: good for verifying everything end-to-end
    TRAIN_EXAMPLES = 50_000
    VAL_EXAMPLES   = 2_000
    TOKENIZER_TRAIN_EXAMPLES = 30_000

    SEQ_LEN = 256
    VOCAB_SIZE = 8_000

    D_MODEL = 384
    N_LAYERS = 6
    N_HEADS = 6
    D_FF = 4 * D_MODEL

    DIFFUSION_STEPS = 64

    TRAIN_STEPS = 2_000
    BATCH_SIZE = 32
    GRAD_ACCUM = 1
    LR = 3e-4
    WEIGHT_DECAY = 0.1
    WARMUP_STEPS = 200

elif RUN_MODE == "budget_100":
    # Heavier: better quality, uses more compute
    TRAIN_EXAMPLES = 1000_000
    VAL_EXAMPLES   = 10_000
    TOKENIZER_TRAIN_EXAMPLES = 150_000

    SEQ_LEN = 256
    VOCAB_SIZE = 26_000

    D_MODEL = 512
    N_LAYERS = 10
    N_HEADS = 8
    D_FF = 4 * D_MODEL

    DIFFUSION_STEPS = 128

    TRAIN_STEPS = 200_000
    BATCH_SIZE = 32
    GRAD_ACCUM = 2
    LR = 2e-4
    WEIGHT_DECAY = 0.1
    WARMUP_STEPS = 1_000

else:
    raise ValueError("RUN_MODE must be 'quick' or 'budget_100'")

print("RUN_MODE:", RUN_MODE)

RUN_MODE: budget_100


In [43]:
class Channels():
    def AWGN(self, Tx_sig, n_var):
        Rx_sig = torch.randn_like(Tx_sig) * n_var
        return Rx_sig

    def Rayleigh(self, Tx_sig, n_var):
        shape = Tx_sig.shape
        H_real = torch.normal(0, math.sqrt(1 / 2), size=[1]).to(Tx_sig.device)
        H_imag = torch.normal(0, math.sqrt(1 / 2), size=[1]).to(Tx_sig.device)
        H = torch.Tensor([[H_real, -H_imag], [H_imag, H_real]]).to(Tx_sig.device)
        Tx_sig = torch.matmul(Tx_sig.view(shape[0], -1, 2), H)
        Rx_sig = self.AWGN(Tx_sig, n_var)
        Rx_sig = torch.matmul(Rx_sig, torch.inverse(H)).view(shape)
        return Rx_sig

    def Rician(self, Tx_sig, n_var, K=1):
        shape = Tx_sig.shape
        mean = math.sqrt(K / (K + 1))
        std = math.sqrt(1 / (K + 1))
        H_real = torch.normal(mean, std, size=[1]).to(Tx_sig.device)
        H_imag = torch.normal(mean, std, size=[1]).to(Tx_sig.device)
        H = torch.Tensor([[H_real, -H_imag], [H_imag, H_real]]).to(Tx_sig.device)
        Tx_sig = torch.matmul(Tx_sig.view(shape[0], -1, 2), H)
        Rx_sig = self.AWGN(Tx_sig, n_var)
        Rx_sig = torch.matmul(Rx_sig, torch.inverse(H)).view(shape)
        return Rx_sig

    def forward(self, Tx_sig, n_var, channel_type="AWGN", K=1):
        if channel_type == "AWGN":
            return self.AWGN(Tx_sig, n_var)
        elif channel_type == "Rayleigh":
            return self.Rayleigh(Tx_sig, n_var)
        elif channel_type == "Rician":
            return self.Rician(Tx_sig, n_var, K=K)
        elif channel_type is None or channel_type == "None":
            return Tx_sig
        else:
            raise ValueError(f"Unknown channel_type: {channel_type}")

In [16]:
print('Encoding sentences...')
encoded_batch = tokenizer.encode_batch(sentences)
results = [encoding.ids for encoding in encoded_batch]

# 80/10/10 Train-Val-Test split
train_split_idx = round(len(results) * 0.8)
test_split_idx = round(len(results)*0.9)
train_data = results[:train_split_idx]
val_data = results[train_split_idx:test_split_idx]
test_data = results[test_split_idx:]



Encoding sentences...


In [17]:
from torch.utils.data import IterableDataset, DataLoader

class TokenBlockDataset(IterableDataset):
    def __init__(self, tokenized_ds, seq_len, shuffle=False, seed=0):
        """
        Args:
            tokenized_ds: A list of lists containing token IDs (e.g., train_data)
            seq_len: The desired sequence length for each block
            shuffle: Whether to shuffle the sentences
            seed: Random seed for shuffling
        """
        self.tokenized_ds = tokenized_ds
        self.seq_len = seq_len
        self.shuffle = shuffle
        self.seed = seed

    def __iter__(self):
        indices = list(range(len(self.tokenized_ds)))
        if self.shuffle:
            rng = random.Random(self.seed)
            rng.shuffle(indices)

        buffer = []
        for idx in indices:
            # Grab the pre-tokenized ID list directly
            ids = self.tokenized_ds[idx]
            buffer.extend(ids)

            # Chunk into blocks of seq_len
            while len(buffer) >= self.seq_len:
                block = buffer[:self.seq_len]
                buffer = buffer[self.seq_len:]
                yield torch.tensor(block, dtype=torch.long)

train_blocks = TokenBlockDataset(train_data, SEQ_LEN, shuffle=True, seed=10)
val_blocks   = TokenBlockDataset(val_data,   SEQ_LEN, shuffle=False)
test_blocks  = TokenBlockDataset(test_data,  SEQ_LEN, shuffle=False)
PAD_ID = hf_tokenizer.convert_tokens_to_ids("<PAD>")

def collate_blocks(batch):
    input_ids = torch.stack(batch, dim=0)  # [B, L]
    attention_mask = (input_ids != PAD_ID)
    return {"input_ids": input_ids, "attention_mask": attention_mask}
    
train_loader = DataLoader(train_blocks, batch_size=BATCH_SIZE, collate_fn=collate_blocks)
val_loader   = DataLoader(val_blocks,batch_size=BATCH_SIZE, collate_fn=collate_blocks)
test_loader = DataLoader(test_blocks,batch_size=BATCH_SIZE, collate_fn=collate_blocks)
# Check a batch
b = next(iter(train_loader))
print({k: v.shape for k, v in b.items()})
print("Decoded snippet:\n", hf_tokenizer.decode(b["input_ids"][0][:120].tolist()))

{'input_ids': torch.Size([32, 256]), 'attention_mask': torch.Size([32, 256])}
Decoded snippet:
 <START> mt some member states have unilaterally decided to introduce legislation preventing europeans from using internet gambling sites run by companies registered in other european union countries . <END> <START> the epp ed group has insisted throughout that the resolution should concentrate on practical proposals that would reinforce consumer safety without delay . <END> <START> colleagues please be quiet . our debate is not yet over . <END> <START> the rapporteur mr viola has asked for the floor to speak on block . <END> <START> . the banks will also be tested on the basis of an upper capital threshold core tier one . <END> <START> it is very significant that the debate on this issue should have aroused so


In [19]:
# # Example forward pass with channel noise applied after the encoder:
# batch = next(iter(train_loader))
# input_ids = batch["input_ids"].to(device)
# attention_mask = batch["attention_mask"].to(device)  # bool, True = real token
# timesteps = torch.randint(1, cfg.diffusion_steps + 1, (input_ids.size(0),), device=device)

# logits = model(input_ids, timesteps, attention_mask=attention_mask, n_var=0.1, channel_type="AWGN")
# print(logits.shape)  # [B, L, V]

In [60]:
train_data[0:3]

[[1,
  184,
  865,
  6,
  455,
  838,
  8,
  360,
  732,
  12,
  310,
  455,
  364,
  31,
  577,
  4,
  267,
  2165,
  12,
  1978,
  43,
  2],
 [1,
  4,
  565,
  437,
  34,
  122,
  12757,
  211,
  5,
  64,
  26,
  25,
  215,
  31,
  118,
  5,
  10,
  25,
  14,
  453,
  320,
  11,
  17,
  5,
  2],
 [1, 18, 106, 251, 7, 948, 4, 1609, 6, 4, 443, 654, 9, 4, 173, 5, 2]]

In [61]:
print(len(train_data),len(val_data),len(test_data))

58778 7347 7347


In [62]:
def make_sigma_schedule(T, sigma_min=0.01, sigma_max=1.0, device="cuda"):
    # sigma[0] = sigma_min ... sigma[T] = sigma_max, geometric interpolation
    steps = torch.arange(T + 1, device=device).float() / T
    sigmas = sigma_min * (sigma_max / sigma_min) ** steps
    return sigmas  # shape [T+1], index with integer t in [0, T]

In [63]:
class ChannelDiffusionLM(nn.Module):
    def __init__(self, cfg: ChannelDiffusionConfig):
        super().__init__()
        self.cfg = cfg
        self.channels = Channels()

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.seq_len, cfg.d_model)
        self.time_emb = nn.Embedding(cfg.diffusion_steps + 1, cfg.d_model)
        
        # --- ELF Mode Embedding ---
        # Index 0 = Denoising Mode, Index 1 = Decoding/Rounding Mode
        self.mode_emb = nn.Embedding(2, cfg.d_model) 
        
        self.drop = nn.Dropout(cfg.dropout)

        # ELF simplifies the model into a unified backbone rather than a separate enc/dec split
        # but we can preserve your architecture and treat the 'denoiser' as the unified flow engine.
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=cfg.d_model, nhead=cfg.n_heads, dim_feedforward=cfg.d_ff,
                dropout=cfg.dropout, batch_first=True, activation="gelu", norm_first=True,
            ), num_layers=cfg.n_enc_layers
        )
        self.z_ln = nn.LayerNorm(cfg.d_model)

        self.denoiser = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=cfg.d_model, nhead=cfg.n_heads, dim_feedforward=cfg.d_ff,
                dropout=cfg.dropout, batch_first=True, activation="gelu", norm_first=True,
            ), num_layers=cfg.n_dec_layers
        )
        self.ln_f = nn.LayerNorm(cfg.d_model)

        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

        sigmas = make_sigma_schedule(cfg.diffusion_steps, cfg.sigma_min, cfg.sigma_max)
        self.register_buffer("sigmas", sigmas)

    def encode(self, input_ids, attention_mask=None):
        B, L = input_ids.shape
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0)
        x = self.drop(self.tok_emb(input_ids) + self.pos_emb(pos))
        src_key_padding_mask = None if attention_mask is None else ~attention_mask
        x = self.encoder(x, src_key_padding_mask=src_key_padding_mask)
        z0 = self.z_ln(x)
        return z0

    def denoise_step(self, z_t, t, mode_idx=0, attention_mask=None):
        """
        ELF Denoising step taking a mode indicator (0 = Denoise, 1 = Decode/Round).
        """
        B, L, D = z_t.shape
        t_emb = self.time_emb(t).unsqueeze(1)  # [B, 1, D]
        
        # Create and add the Mode Embedding (0 for denoise, 1 for decode)
        modes = torch.full((B,), mode_idx, dtype=torch.long, device=z_t.device)
        m_emb = self.mode_emb(modes).unsqueeze(1)  # [B, 1, D]
        
        # Incorporate continuous-time state, step-time, and target task mode
        x = z_t + t_emb + m_emb
        
        src_key_padding_mask = None if attention_mask is None else ~attention_mask
        x = self.denoiser(x, src_key_padding_mask=src_key_padding_mask)
        z0_hat = self.ln_f(x)
        return z0_hat

    def forward(self, input_ids, t, mode_idx=0, attention_mask=None, channel_type=None):
        ctype = channel_type if channel_type is not None else self.cfg.channel_type
        z0 = self.encode(input_ids, attention_mask)

        # 1. Forward process: add AWGN noise according to the sigma trajectory
        sigma_t = self.sigmas[t].view(-1, 1, 1)  # [B, 1, 1]
        z_t = self.channels.AWGN(z0, sigma_t)

        # 2. Denoiser Step using the ELF mode indicator
        z0_hat = self.denoise_step(z_t, t, mode_idx=mode_idx, attention_mask=attention_mask)
        logits = self.lm_head(z0_hat)
        return logits, z0, z0_hat

In [64]:
def channel_diffusion_loss(model, batch, lambda_ce=1.0):
    device = next(model.parameters()).device
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    B = input_ids.size(0)

    # Pick random timesteps
    t = torch.randint(0, model.cfg.diffusion_steps + 1, (B,), device=device)
    mask = attention_mask.unsqueeze(-1).float()

    # --- ELF Dual-Task Training Scheme ---
    # Randomly decide which items in the batch are optimized for continuous denoising 
    # and which are optimized for decoding back to tokens.
    mode_mask = (torch.rand(B, device=device) < 0.2).long()  # 20% Decoding, 80% Denoising
    
    # 1. Run Denoise-mode forward (mode=0) for continuous latent optimization
    logits_denoise, z0, z0_hat = model(input_ids, t, mode_idx=0, attention_mask=attention_mask)
    mse = ((z0_hat - z0) ** 2 * mask).sum() / mask.sum().clamp_min(1) / z0.size(-1)

    # 2. Run Decode-mode forward (mode=1) to train the rounding layer
    # For decoding mode, we pass a lightly-noised z_t or clean z_0 to teach the network
    # how to project latents back to vocab dimensions.
    logits_decode, _, _ = model(input_ids, t, mode_idx=1, attention_mask=attention_mask)
    
    targets = input_ids.clone()
    targets[~attention_mask] = -100
    
    ce = F.cross_entropy(logits_decode.view(-1, logits_decode.size(-1)), targets.view(-1), ignore_index=-100)

    # Total Joint Optimization
    loss = mse + lambda_ce * ce
    return loss, {"mse": mse.item(), "ce": ce.item(), "t_mean": t.float().mean().item()}

In [65]:
@torch.no_grad()
def reverse_sample(model, z_T, attention_mask=None, n_steps=None):
    T = model.cfg.diffusion_steps
    n_steps = n_steps or T
    B = z_T.size(0)
    z_t = z_T
    ts = torch.linspace(T, 0, n_steps + 1, device=z_T.device).long()

    # 1. Traversal completely in Continuous Space (Denoise Mode = 0)
    for i in range(n_steps):
        t_cur = ts[i].expand(B)
        
        # We step exclusively in mode 0
        z0_hat = model.denoise_step(z_t, t_cur, mode_idx=0, attention_mask=attention_mask)
        
        if i == n_steps - 1:
            z_t = z0_hat
            break
            
        t_next = ts[i + 1].expand(B)
        sigma_next = model.sigmas[t_next].view(-1, 1, 1)
        z_t = model.channels.AWGN(z0_hat, sigma_next)

    # 2. Last-Mile Discretization (Decode Mode = 1)
    # We switch to mode_idx=1 so the network maps the final clean continuous latents back to words.
    t_zero = torch.zeros(B, dtype=torch.long, device=z_T.device)
    z_final = model.denoise_step(z_t, t_zero, mode_idx=1, attention_mask=attention_mask)
    
    logits = model.lm_head(z_final)
    token_ids = logits.argmax(-1)
    return token_ids, logits

In [66]:
cfg = ChannelDiffusionConfig(
    vocab_size=len(hf_tokenizer),
    seq_len=SEQ_LEN,
    d_model=D_MODEL,
    n_enc_layers=N_LAYERS // 2,
    n_dec_layers=N_LAYERS // 2,
    n_heads=N_HEADS,
    d_ff=D_FF,
    dropout=0.1,
    diffusion_steps=DIFFUSION_STEPS,
    sigma_min=0.01,
    sigma_max=1.0,     # calibrate against z0's actual std — see note below
    channel_type="AWGN",
)
model = ChannelDiffusionLM(cfg).to(device)

# training step
loss, stats = channel_diffusion_loss(model, batch)
loss.backward()

/tmp/ipykernel_58/3428711105.py:19: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
/tmp/ipykernel_58/3428711105.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.denoiser = nn.TransformerEncoder(


In [67]:
# ----------------------------------------------------------------------
# Config for the training run itself
# ----------------------------------------------------------------------
TOTAL_STEPS = 20000        # <-- set this directly instead of NUM_EPOCHS * len(train_loader)
LR = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 500
GRAD_CLIP = 1.0
LOG_EVERY = 50
EVAL_EVERY = 500
SAMPLE_EVERY = 1000
LAMBDA_CE = 1.0
CKPT_PATH = "channel_diffusion_ckpt.pt"

device = next(model.parameters()).device

def make_lr_lambda(warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))
    return lr_lambda

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = LambdaLR(optimizer, lr_lambda=make_lr_lambda(WARMUP_STEPS, TOTAL_STEPS))

In [68]:
@torch.no_grad()
def evaluate(model, val_loader, n_batches=20):
    model.eval()
    total_loss, total_mse, total_ce, n = 0.0, 0.0, 0.0, 0
    for i, batch in enumerate(val_loader):
        if i >= n_batches:
            break
        batch = {k: v.to(device) for k, v in batch.items()}
        loss, stats = channel_diffusion_loss(model, batch, lambda_ce=LAMBDA_CE)
        total_loss += loss.item()
        total_mse += stats["mse"]
        total_ce += stats["ce"]
        n += 1
    model.train()
    return {"val_loss": total_loss / n, "val_mse": total_mse / n, "val_ce": total_ce / n}


In [69]:
@torch.no_grad()
def sanity_check_sample(model, batch, tokenizer, n_show=2):
    model.eval()
    batch = {k: v.to(device) for k, v in batch.items()}
    input_ids = batch["input_ids"][:n_show]
    attention_mask = batch["attention_mask"][:n_show]

    z0 = model.encode(input_ids, attention_mask)
    T = model.cfg.diffusion_steps
    sigma_T = model.sigmas[T]
    z_T = model.channels.AWGN(z0, sigma_T)

    pred_ids, _ = reverse_sample(model, z_T, attention_mask=attention_mask)

    for i in range(n_show):
        gt = tokenizer.decode(input_ids[i][attention_mask[i]], skip_special_tokens=True)
        pred = tokenizer.decode(pred_ids[i][attention_mask[i]], skip_special_tokens=True)
        print(f"  GT:   {gt}")
        print(f"  PRED: {pred}")
        print()
    model.train()

In [ ]:
def train(model, train_loader, val_loader, tokenizer, total_steps):
    step = 0
    best_val_loss = float("inf")
    model.train()

    train_iter = iter(train_loader)
    start_time = time.time()

    while step < total_steps:
        try:
            batch = next(train_iter)
        except StopIteration:
            # IterableDataset exhausted (e.g. one pass through a finite stream) -> restart it
            train_iter = iter(train_loader)
            batch = next(train_iter)

        batch = {k: v.to(device) for k, v in batch.items()}

        loss, stats = channel_diffusion_loss(model, batch, lambda_ce=LAMBDA_CE)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        if step % LOG_EVERY == 0:
            lr_now = scheduler.get_last_lr()[0]
            elapsed = time.time() - start_time
            print(f"step {step}/{total_steps} | loss {loss.item():.4f} "
                  f"(mse {stats['mse']:.4f}, ce {stats['ce']:.4f}) "
                  f"| t_mean {stats['t_mean']:.1f} | lr {lr_now:.2e} | {elapsed:.1f}s")

        if step % EVAL_EVERY == 0 and step > 0:
            val_stats = evaluate(model, val_loader)
            print(f"  [eval] step {step} | val_loss {val_stats['val_loss']:.4f} "
                  f"(mse {val_stats['val_mse']:.4f}, ce {val_stats['val_ce']:.4f})")

            if val_stats["val_loss"] < best_val_loss:
                best_val_loss = val_stats["val_loss"]
                torch.save({
                    "model_state_dict": model.state_dict(),
                    "cfg": model.cfg,
                    "step": step,
                    "val_loss": best_val_loss,
                }, CKPT_PATH)
                print(f"  [ckpt] saved new best (val_loss={best_val_loss:.4f}) -> {CKPT_PATH}")

        if step % SAMPLE_EVERY == 0 and step > 0:
            print(f"  [sample] step {step} reverse-diffusion sanity check:")
            sample_batch = next(iter(val_loader))
            sanity_check_sample(model, sample_batch, tokenizer)

        step += 1

    print("training complete.")


train(model, train_loader, val_loader, hf_tokenizer, total_steps=TOTAL_STEPS)

step 0/20000 | loss 94.8934 (mse 1.9993, ce 92.8941) | t_mean 70.2 | lr 6.00e-07 | 0.8s
step 50/20000 | loss 58.1129 (mse 1.0104, ce 57.1024) | t_mean 79.2 | lr 3.06e-05 | 68.3s
step 100/20000 | loss 34.8263 (mse 0.3872, ce 34.4391) | t_mean 55.0 | lr 6.06e-05 | 134.4s
step 150/20000 | loss 26.6856 (mse 0.1471, ce 26.5385) | t_mean 63.1 | lr 9.06e-05 | 200.2s
step 200/20000 | loss 22.3714 (mse 0.0641, ce 22.3074) | t_mean 66.2 | lr 1.21e-04 | 266.2s
step 250/20000 | loss 18.4956 (mse 0.0478, ce 18.4479) | t_mean 58.8 | lr 1.51e-04 | 332.6s
step 300/20000 | loss 15.8921 (mse 0.0385, ce 15.8536) | t_mean 64.3 | lr 1.81e-04 | 398.5s
step 350/20000 | loss 13.9559 (mse 0.0355, ce 13.9203) | t_mean 60.8 | lr 2.11e-04 | 464.3s
step 400/20000 | loss 12.5300 (mse 0.0332, ce 12.4968) | t_mean 65.3 | lr 2.41e-04 | 530.4s
step 450/20000 | loss 11.5902 (mse 0.0330, ce 11.5572) | t_mean 53.4 | lr 2.71e-04 | 596.8s


In [ ]:
def print_model_parameters(model: torch.nn.Module):
    """
    Prints a detailed breakdown of the model's parameters,
    taking into account weight sharing (like the tied LM head).
    """
    total_params = 0
    trainable_params = 0
    unique_params_set = set()

    print(f"{'Module Name':<45} | {'Shape':<20} | {'Parameters':<15} | {'Requires Grad'}")
    print("-" * 95)

    for name, param in model.named_parameters():
        param_id = id(param)
        is_shared = param_id in unique_params_set
        unique_params_set.add(param_id)
        
        num_params = param.numel()
        grad_status = str(param.requires_grad)
        shape_str = str(list(param.shape))
        
        # We flag shared weights (like lm_head.weight tied to tok_emb.weight)
        display_name = f"{name} (Shared)" if is_shared else name
        print(f"{display_name:<45} | {shape_str:<20} | {num_params:<15,} | {grad_status}")
        
        if not is_shared:
            total_params += num_params
            if param.requires_grad:
                trainable_params += num_params

    print("-" * 95)
    print(f"Total Unique Parameters:     {total_params:,}")
    print(f"Total Trainable Parameters:  {trainable_params:,}")

In [ ]:
print_model_parameters(model)